# AdaBelief baseline column - vision, finance, language (old protocol, direct comparison)

Turn on **GPU (T4 x2)** and **Internet**, then *Run All*.

These are the existing part-A scripts with only AdaBelief added as an optimizer (`eps=1e-16`, decoupled weight decay 1e-4, no rectification,
bit-identical to `adabelief_pytorch 0.2.1` in a float64 test): same data and splits, same models, 5000 steps, batch 32, cosine to 0.1*lr,
clip 5, same seeds 42/43/44, same LR grid `[1e-4,3e-4,1e-3,3e-3]` with the 100-step search and the same tuning subsets, same Hutchinson probes
(so wall time and Hessian statistics are comparable), same aggregation. Nothing else is run (no AdamW, no ablation).

GPU 0 runs finance and then vision, GPU 1 runs language (with one GPU they run one after the other, so the `wall` metric is never measured under contention).
Finance uses the **same 100 tickers** as your existing runs if a `run1a_checkpoint.json` is attached as input data (otherwise the legacy rule recomputes them and a warning says so).

**Time budget / resume.** Each script stops cleanly after 9 h 55 min, saves its checkpoint and writes `/kaggle/working/evie_e1_<domain>_pack.zip`.
To continue: download the zips, start a new session, *Add Data* -> upload them (or attach this notebook's previous Output), turn on GPU + Internet, *Run All* again.
The logs say `[SESSION n] RESUMED FROM CHECKPOINT: k/N units were already complete`; the LR search is not repeated and finished units are not rerun.

In [ ]:
# ---- 1. install the embedded code (nothing to upload or clone) -------------------------------------------------
import base64, io, os, sys, subprocess, tarfile, time, glob, json, threading
BUNDLE = """
...
"""
ON_KAGGLE = os.path.isdir("/kaggle/working")
ROOT = "/kaggle/working/repo" if ON_KAGGLE else os.path.abspath("./repo_nb")
os.makedirs(ROOT, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as _t:
    try:
        _t.extractall(ROOT, filter="data")
    except TypeError:                      # Python < 3.12
        _t.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, ROOT)
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance", "datasets"], check=False)

def sh(cmd, env=None, prefix=""):
    """Run a shell command, stream its output live, return (returncode, captured_text)."""
    e = dict(os.environ); e.update({k: str(v) for k, v in (env or {}).items()})
    if cmd.startswith("python "):
        cmd = '"' + sys.executable + '" ' + cmd[len("python "):]
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e, cwd=ROOT)
    buf = []
    for line in p.stdout:
        print(prefix + line, end="", flush=True); buf.append(line)
    p.wait()
    return p.returncode, "".join(buf)

try:
    import torch
    N_GPU = torch.cuda.device_count()
    print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
except Exception as _e:
    N_GPU = 0; print("torch not importable:", _e)
if ON_KAGGLE and N_GPU == 0:
    raise SystemExit("No GPU found: turn on 'Accelerator: GPU T4 x2' in the notebook settings and run again.")
if ON_KAGGLE:
    import urllib.request
    try:
        urllib.request.urlopen("https://pypi.org", timeout=10)
    except Exception:
        raise SystemExit("No internet: turn on 'Internet' in the notebook settings and run again.")

In [ ]:
# ---- 2. settings --------------------------------------------------------------------------------------------------
BUDGET_MIN = 595            # 9 h 55 min per script
ELAPSED_OFFSET_MIN = 0      # minutes of this Kaggle session already used before running this notebook
SCRIPTS = {"vision": "vision/benchmark/run1_partD_adabelief.py",
           "finance": "finance/benchmark/run1_partD_adabelief.py",
           "language": "language/benchmark/language_run1_partd_adabelief.py"}
old_tick = sorted(glob.glob("/kaggle/input/**/run1a_checkpoint.json", recursive=True))
BASE_ENV = {"EVIE_TIME_BUDGET_MIN": BUDGET_MIN}
if old_tick:
    BASE_ENV["EVIE_OLD_TICKERS"] = old_tick[0]
    print("finance: using the ticker list of the existing runs:", old_tick[0])
else:
    print("[WARN] finance: no run1a_checkpoint.json under /kaggle/input -> tickers are recomputed with the legacy rule (seed 1234); "
          "attach the dataset with your old run1a_checkpoint.json for the identical list.")

In [ ]:
# ---- 3. run (streams the logs live; a [HEARTBEAT] line every 10 min) -----------------------------------------------------
T0 = time.time()
def run_script(domain, gpu):
    env = dict(BASE_ENV); env["EVIE_DEVICE"] = f"cuda:{gpu}"
    env["EVIE_ELAPSED_OFFSET_MIN"] = ELAPSED_OFFSET_MIN + (time.time() - T0) / 60.0
    tag = {"vision": "VIS", "finance": "FIN", "language": "LM "}[domain]
    rc, _ = sh(f"python {SCRIPTS[domain]}", env=env, prefix=f"[{tag}] ")
    return rc

def launch_all():
    codes = {}
    if N_GPU >= 2:
        def gpu0():
            codes["finance"] = run_script("finance", 0)
            codes["vision"] = run_script("vision", 0)
        def gpu1():
            codes["language"] = run_script("language", 1)
        ts = [threading.Thread(target=gpu0), threading.Thread(target=gpu1)]
        [t.start() for t in ts]; [t.join() for t in ts]
    else:
        for d in ("finance", "language", "vision"):
            codes[d] = run_script(d, 0)
    return codes

CODES = launch_all()
print("\nexit codes:", CODES, "(0 = finished or stopped cleanly at the budget; see the log lines [SESSION END] for 'N left')")

In [ ]:
# ---- 4. summary (read from the checkpoints) and, if your old checkpoints are attached, the comparison ------------------------
OUTD = {d: (f"/kaggle/working/evie_e1_{d}" if ON_KAGGLE else None) for d in SCRIPTS}
CK = {"vision": "vision_run1d_adabelief_checkpoint.json", "finance": "run1d_adabelief_checkpoint.json", "language": "lm_run1d_adabelief_checkpoint.json"}
PRIM = {"vision": "top1", "finance": "test_mse", "language": "valloss"}
import numpy as np, math
for d in SCRIPTS:
    p = os.path.join(OUTD[d] or ".", CK[d])
    if not os.path.exists(p):
        print(d, ": no checkpoint found"); continue
    ck = json.load(open(p)); done, tot = len(ck.get("completed", {})), None
    vals = []
    for k, v in ck["raw"].items():
        runs = [v] if d == "vision" else v
        x = [r[PRIM[d]] for r in runs if not (isinstance(r[PRIM[d]], float) and math.isnan(r[PRIM[d]]))]
        if x: vals.append(float(np.median(x)))
    print(f"{d:9s} units complete {done}, sessions {len(ck.get('sessions', []))}, failed {list(ck.get('failed', {}))[:3]}  "
          f"{PRIM[d]} = {np.mean(vals):.5g} +/- {np.std(vals):.3g} (n={len(vals)})" if vals else f"{d}: nothing finished yet")

OLD = {"vision": "vision_run1{x}_checkpoint.json", "finance": "run1{x}_checkpoint.json", "language": "lm_run1{x}_checkpoint.json"}
for d in SCRIPTS:
    refs = {}
    for lab in "abc":
        f = sorted(glob.glob("/kaggle/input/**/" + OLD[d].format(x=lab), recursive=True))
        if f: refs[lab.upper()] = f[0]
    p = os.path.join(OUTD[d] or ".", CK[d])
    if refs and os.path.exists(p):
        print(f"\n=== comparison {d} against {refs}")
        sh(f"python -m evie_exp.analysis.compare_adabelief --domain {d} --adabelief {p} --existing " + " ".join(f"{k}={v}" for k, v in refs.items()) +
           (f" --out /kaggle/working/tables" if ON_KAGGLE else " --out tables"))
print("\nFiles to keep: /kaggle/working/evie_e1_<domain>/ (checkpoints, logs, csv/json) and the three evie_e1_*_pack.zip")